# 06. Parquet から Iceberg への変換

既存の Parquet ファイルを Iceberg テーブルにする方法は、大きく 2 つあります。

| 方式 | 方法 | 仕組み |
| --- | --- | --- |
| **書き直す** | Spark の CTAS | Parquet を読み、Iceberg テーブルとして新しいファイルに書き直す |
| **書き直さない** | PyIceberg の `add_files` | 既存の Parquet ファイルをそのままテーブルに登録する（メタデータだけ作る） |

このノートブックでは両方を試し、できあがるファイルを比べます。

- テーブル: `handson.taxi_ctas`（CTAS）、`handson.taxi_addfiles`（add_files）
- 事前に `make data` でデータを取得しておく

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("06_parquet_conversion").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. CTAS（書き直す方式）

`CREATE TABLE ... AS SELECT` で、Parquet ファイルを読んだ結果から Iceberg テーブルを作ります。
スキーマは SELECT の結果から決まり、データは Iceberg の管理下（テーブルの `data/`）に新しいファイルとして書かれます。

In [ ]:
sql("DROP TABLE IF EXISTS handson.taxi_ctas PURGE")
sql("""
CREATE TABLE handson.taxi_ctas
USING iceberg
AS SELECT * FROM parquet.`/workspace/data/yellow_tripdata_2024-12.parquet`
""")
sql("SELECT count(*) AS data_files, sum(record_count) AS rows, sum(file_size_in_bytes) AS bytes FROM handson.taxi_ctas.files")
sql("SELECT file_path FROM handson.taxi_ctas.files")

元の Parquet ファイルとは別に、Iceberg が付けた名前のファイルができています。
元のファイルはそのまま残るので、変換後は二重にストレージを使います。その代わり、ファイルサイズや圧縮方式は Iceberg の設定どおりに揃います。

## 2. add_files（書き直さない方式）

PyIceberg で次の順に進めます。

1. Parquet のスキーマから Iceberg テーブルを作る
2. Parquet ファイルをテーブルの場所（RustFS）にアップロードする
3. `add_files` で、アップロードしたファイルをテーブルに登録する

PyIceberg の接続設定は環境変数で渡してあるので、`load_catalog("lakehouse")` だけで Polaris に接続できます。

In [ ]:
import pyarrow.parquet as pq
from pyiceberg.catalog import load_catalog

catalog = load_catalog("lakehouse")
source = "/workspace/data/yellow_tripdata_2025-01.parquet"

if catalog.table_exists("handson.taxi_addfiles"):
    catalog.purge_table("handson.taxi_addfiles")

# 1. Parquet のスキーマ（列名と型）をそのまま使ってテーブルを作る
table = catalog.create_table("handson.taxi_addfiles", schema=pq.read_schema(source))
print(table.location())

アップロードには、テーブルの `io`（Polaris から払い出された一時的な S3 認証情報を使うファイル入出力）を使います。
この認証情報で書き込めるのは、このテーブルの場所の下だけです。

In [ ]:
# 2. テーブルの場所の下にアップロードする（ファイル名は元のまま）
target = f"{table.location()}/data/imported/yellow_tripdata_2025-01.parquet"
with open(source, "rb") as src, table.io.new_output(target).create(overwrite=True) as dst:
    dst.write(src.read())

# 3. ファイルを書き直さずにテーブルに登録する
table.add_files([target])
print(table.current_snapshot().summary.additional_properties)

## 3. 比べる

Spark から両方のテーブルを見てみます。
`taxi_addfiles` のデータファイルは、アップロードした元のファイル（名前もサイズもそのまま）が 1 つだけです。

In [ ]:
sql("SELECT count(*) AS data_files, sum(record_count) AS rows, sum(file_size_in_bytes) AS bytes FROM handson.taxi_addfiles.files")
sql("SELECT file_path FROM handson.taxi_addfiles.files")

import os
print("元のファイルのサイズ:", os.path.getsize(source), "bytes")

どちらも普通の Iceberg テーブルとしてクエリできます。

In [ ]:
sql("""
SELECT 'ctas' AS source, count(*) AS trips, round(avg(total_amount), 2) AS avg_total FROM handson.taxi_ctas
UNION ALL
SELECT 'add_files', count(*), round(avg(total_amount), 2) FROM handson.taxi_addfiles
""")

## 注意点

- `add_files` はスキーマを検証しません。テーブルの列と型が Parquet ファイルと合っていることを、登録する側が保証する必要があります
- 登録したファイルは、以後 Iceberg の管理下に入ります。07 のメンテナンス（スナップショットの失効など）で不要と判断されると削除されることがあります
- 元の Parquet には Iceberg の列 ID がないので、列名で対応を取る設定（name mapping）が自動で付きます

## まとめ

| | CTAS | add_files |
| --- | --- | --- |
| データの書き直し | する（時間とストレージを使う） | しない（速い） |
| ファイルの形 | Iceberg の設定どおりに揃う | 元のまま |
| スキーマ | SELECT の結果から決まる | 登録する側が合わせる |
| 向いている場面 | 形を整えたい、パーティションを切り直したい | 大量の既存ファイルをすばやく取り込みたい |